# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pakizahassan/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


### Ranked Content Action Queue

I convert the model's decline-risk scores into a ranked review queue rather than treating the predictions as automatic content decisions.

The queue uses three action levels:

- **Review First** — pages with the strongest measured decline-risk signal.
- **Monitor** — pages with moderate risk that should be watched before taking action.
- **Lower Priority** — pages with weaker measured risk signals.

Each page also receives human-readable reason codes based on its historical search and engagement signals.

The reason codes are directional decision-support cues. They do not prove that a page needs a refresh or that changing it will improve search performance.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

# ---------------------------
# 1. Load only needed fields
# ---------------------------

columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "scroll_events"
]

df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/data_0.parquet",
    columns=columns
)

df["report_date"] = pd.to_datetime(df["report_date"])

# Time-aware windows used in the capstone
past = df[df["report_date"] <= "2026-03-21"].copy()
future = df[df["report_date"] >= "2026-03-22"].copy()

print("Historical rows:", len(past))
print("Future rows:", len(future))


Historical rows: 6573997
Future rows: 3267381


In [2]:
# ---------------------------
# 2. Historical features
# ---------------------------

past_agg = (
    past.groupby(
        ["client_hash_id", "content_hash_id"],
        observed=True
    )
    .agg(
        past_days=("report_date", "nunique"),
        past_impressions=("gsc_impressions", "sum"),
        past_clicks=("gsc_clicks", "sum"),
        past_avg_position=("gsc_avg_position", "mean"),
        past_sessions=("ga4_sessions", "sum"),
        past_engaged_sessions=("ga4_engaged_sessions", "sum"),
        past_scroll_events=("scroll_events", "sum")
    )
    .reset_index()
)

past_agg["past_daily_impressions"] = (
    past_agg["past_impressions"] /
    past_agg["past_days"]
)

past_agg["past_ctr"] = np.where(
    past_agg["past_impressions"] > 0,
    past_agg["past_clicks"] / past_agg["past_impressions"],
    0
)

past_agg["past_daily_sessions"] = (
    past_agg["past_sessions"] /
    past_agg["past_days"]
)

past_agg["past_engagement_rate"] = np.where(
    past_agg["past_sessions"] > 0,
    past_agg["past_engaged_sessions"] /
    past_agg["past_sessions"],
    0
)

past_agg["past_daily_scroll_events"] = (
    past_agg["past_scroll_events"] /
    past_agg["past_days"]
)

In [3]:
# ---------------------------
# 3. Future outcome
# ---------------------------

future_agg = (
    future.groupby(
        ["client_hash_id", "content_hash_id"],
        observed=True
    )
    .agg(
        future_days=("report_date", "nunique"),
        future_impressions=("gsc_impressions", "sum")
    )
    .reset_index()
)

future_agg["future_daily_impressions"] = (
    future_agg["future_impressions"] /
    future_agg["future_days"]
)

model_df = past_agg.merge(
    future_agg,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Only pages with measurable historical visibility
model_df = model_df[
    model_df["past_daily_impressions"] > 0
].copy()

model_df["impression_change_pct"] = (
    (
        model_df["future_daily_impressions"] -
        model_df["past_daily_impressions"]
    )
    / model_df["past_daily_impressions"]
) * 100

# Proxy target: future impressions declined by at least 20%
model_df["future_decline"] = (
    model_df["impression_change_pct"] <= -20
).astype(int)

print("Modeling pages:", len(model_df))

display(
    model_df["future_decline"]
    .value_counts()
    .rename(index={0: "No Major Decline", 1: "Future Decline"})
)

Modeling pages: 162300


,count
future_decline,
No Major Decline,101460
Future Decline,60840


In [4]:
# ---------------------------
# 4. Grouped model
# ---------------------------

features = [
    "past_daily_impressions",
    "past_ctr",
    "past_avg_position",
    "past_daily_sessions",
    "past_engagement_rate",
    "past_daily_scroll_events"
]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        model_df["future_decline"],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

# Fill missing search-position values using training data only
position_fill = train_df["past_avg_position"].median()

train_df["past_avg_position"] = (
    train_df["past_avg_position"]
    .fillna(position_fill)
)

test_df["past_avg_position"] = (
    test_df["past_avg_position"]
    .fillna(position_fill)
)

X_train = (
    train_df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype("float32")
)

X_test = (
    test_df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype("float32")
)

y_train = train_df["future_decline"]
y_test = test_df["future_decline"]

model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    random_state=42
)

model.fit(X_train, y_train)

test_df["decline_risk_score"] = (
    model.predict_proba(X_test)[:, 1]
)

print("Pages scored:", len(test_df))

Pages scored: 31824


In [6]:
# ---------------------------
# 5. Action levels
# ---------------------------

REVIEW_THRESHOLD = 0.4388
MONITOR_THRESHOLD = 0.3479

test_df["recommended_action"] = np.select(
    [
        test_df["decline_risk_score"] >= REVIEW_THRESHOLD,
        test_df["decline_risk_score"] >= MONITOR_THRESHOLD
    ],
    [
        "Review First",
        "Monitor"
    ],
    default="Lower Priority"
)

display(
    test_df["recommended_action"]
    .value_counts()
    .rename("Pages")
    .to_frame()
)

,Pages
recommended_action,
Lower Priority,13025
Monitor,11054
Review First,7745


In [7]:
# ---------------------------
# 6. Human-readable reason codes
# ---------------------------

HIGH_IMPRESSIONS = 36.9
WEAK_POSITION = 19.75


def build_reason_codes(row):
    reasons = []

    if row["past_daily_impressions"] >= HIGH_IMPRESSIONS:
        reasons.append("HIGH_VISIBILITY_AT_RISK")

    if row["past_ctr"] <= 0:
        reasons.append("LOW_CLICK_CAPTURE")

    if row["past_avg_position"] >= WEAK_POSITION:
        reasons.append("WEAK_SEARCH_POSITION")

    if row["past_engagement_rate"] <= 0:
        reasons.append("LOW_ENGAGEMENT")

    if row["decline_risk_score"] >= REVIEW_THRESHOLD:
        reasons.append("DECLINE_RISK_SIGNAL")

    if not reasons:
        reasons.append("NO_STRONG_REASON_CODE")

    return " | ".join(reasons)


test_df["reason_codes"] = test_df.apply(
    build_reason_codes,
    axis=1
)

In [8]:
ranked_queue = (
    test_df[
        [
            "client_hash_id",
            "content_hash_id",
            "decline_risk_score",
            "recommended_action",
            "reason_codes",
            "past_daily_impressions",
            "past_ctr",
            "past_avg_position",
            "past_engagement_rate"
        ]
    ]
    .sort_values(
        "decline_risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)

ranked_queue.index = ranked_queue.index + 1
ranked_queue.index.name = "priority_rank"

display(ranked_queue.head(20))

,client_hash_id,content_hash_id,decline_risk_score,recommended_action,reason_codes,past_daily_impressions,past_ctr,past_avg_position,past_engagement_rate
priority_rank,,,,,,,,,
1,client_3f0ce4d44fe94f3d,content_e186467e031ce88d,0.786740,Review First,LOW_CLICK_CAPTURE | WEAK_SEARCH_POSITION | LOW...,0.052632,0.0,29.0,0.0
2,client_3f0ce4d44fe94f3d,content_d16ccdd45b7c48f7,0.786740,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,10.0,0.0
3,client_3f0ce4d44fe94f3d,content_ec8fd442962b9bce,0.786740,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,10.0,0.0
4,client_3f0ce4d44fe94f3d,content_25d7a0a4aaca8746,0.786740,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,11.0,0.0
5,client_3f0ce4d44fe94f3d,content_8c087c4643248bc2,0.786740,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,9.0,0.0
6,client_3f0ce4d44fe94f3d,content_cf88a82c77fd99ad,0.786740,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,9.0,0.0
7,client_3f0ce4d44fe94f3d,content_39cac2a87e36bac3,0.786740,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,11.0,0.0
8,client_3f0ce4d44fe94f3d,content_a905934677d7d92e,0.786740,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,11.0,0.0
9,client_3f0ce4d44fe94f3d,content_546d4dbcb6ac7121,0.786740,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,9.0,0.0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use

This playbook is designed for a content or SEO team that needs to decide which pages should be reviewed first.

The ranked queue can be used to:

- prioritize pages for manual content review;
- identify pages showing stronger measured decline-risk signals;
- explain why a page entered the queue through reason codes;
- separate high-priority review candidates from pages that can be monitored;
- support planning when the team cannot manually review every page.

The output is **decision-support**, not an automatic publishing system.

### Limits

The recommendations have several important limits:

- The target is a proxy based on a future impressions decline of at least 20%; it is not a ground-truth label saying a page needs a refresh.
- The analysis uses only March 2026 data, so the observation and outcome windows are short.
- The study is observational. It does not prove that refreshing a recommended page will improve rankings, traffic, or engagement.
- The model was evaluated on held-out clients, but performance may differ for other clients, topics, or future periods.
- GA4 measurement coverage is limited for many pages, so engagement-based reason codes should be interpreted cautiously.
- A high decline-risk score means **review first**, not **automatically change the page**.
- Search demand, seasonality, technical problems, indexing changes, and external events may also explain performance changes.

The queue should therefore be used as a directional prioritization tool followed by human investigation.

In [9]:
# Basic scope and validity checks for the playbook

scope_summary = pd.DataFrame({
    "Item": [
        "Historical feature window",
        "Future outcome window",
        "Held-out pages scored",
        "Training clients",
        "Held-out clients",
        "Client overlap",
        "Target definition",
        "Primary intended use"
    ],
    "Value": [
        "2026-03-01 to 2026-03-21",
        "2026-03-22 to 2026-03-31",
        f"{len(test_df):,}",
        train_df["client_hash_id"].nunique(),
        test_df["client_hash_id"].nunique(),
        len(
            set(train_df["client_hash_id"]) &
            set(test_df["client_hash_id"])
        ),
        "Future daily impressions decline >= 20%",
        "Human review prioritization"
    ]
})

display(scope_summary)


,Item,Value
0,Historical feature window,2026-03-01 to 2026-03-21
1,Future outcome window,2026-03-22 to 2026-03-31
2,Held-out pages scored,"31,824"
3,Training clients,36
4,Held-out clients,9
5,Client overlap,0
6,Target definition,Future daily impressions decline >= 20%
7,Primary intended use,Human review prioritization


In [12]:
# Confirm that recommendation labels are prioritization categories

action_share = (
    test_df["recommended_action"]
    .value_counts()
    .rename_axis("Action")
    .reset_index(name="Pages")
)

action_share["Share (%)"] = (
    action_share["Pages"] / len(test_df) * 100
).round(2)

display(action_share)

,Action,Pages,Share (%)
0,Lower Priority,13025,40.93
1,Monitor,11054,34.73
2,Review First,7745,24.34


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Before Action

A person should review every high-priority page before making a content change.

Before acting, the reviewer should check:

- whether the page is still relevant to the current business or search intent;
- whether the decline may be caused by seasonality or changing search demand;
- whether technical or indexing issues could explain the performance change;
- whether the page already contains accurate and current information;
- whether the page has enough historical visibility to justify intervention;
- whether low engagement reflects missing GA4 coverage rather than weak content;
- whether the recommended action is consistent with the page's role and topic.

The model should be treated as a screening tool that helps decide **where to look first**, not as the final decision-maker.

### No-Go List

The following actions should never be automated directly from the model score:

- automatically deleting a page;
- automatically rewriting or replacing published content;
- automatically changing titles, metadata, or URLs;
- automatically merging pages;
- automatically publishing generated content;
- automatically assuming that a decline was caused by stale content;
- automatically treating low engagement as poor content when analytics coverage may be incomplete.

Any material content change should require human approval.

In [13]:
# Human-review checklist and no-go controls

review_controls = pd.DataFrame({
    "Check": [
        "Search demand / seasonality",
        "Technical or indexing issue",
        "Content accuracy and freshness",
        "Historical visibility",
        "Analytics coverage",
        "Page purpose and intent",
        "Human approval before material change"
    ],
    "Required": [
        True,
        True,
        True,
        True,
        True,
        True,
        True
    ]
})

display(review_controls)

,Check,Required
0,Search demand / seasonality,True
1,Technical or indexing issue,True
2,Content accuracy and freshness,True
3,Historical visibility,True
4,Analytics coverage,True
5,Page purpose and intent,True
6,Human approval before material change,True


In [14]:
no_go_actions = pd.DataFrame({
    "Automated Action": [
        "Delete page",
        "Rewrite page",
        "Change URL",
        "Merge content",
        "Publish content",
        "Assume refresh is required"
    ],
    "Allowed Automatically": [
        False,
        False,
        False,
        False,
        False,
        False
    ]
})

display(no_go_actions)

,Automated Action,Allowed Automatically
0,Delete page,False
1,Rewrite page,False
2,Change URL,False
3,Merge content,False
4,Publish content,False
5,Assume refresh is required,False


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and Retrain Triggers

The recommendation system should be monitored because search behavior, content performance, and client mix can change over time.

The queue should be reviewed or the model retrained when one or more of the following occurs:

- prediction quality drops meaningfully on new evaluation data;
- precision or recall falls below the level observed in the current validation;
- the share of pages assigned to each action category changes substantially;
- feature distributions shift compared with the March 2026 training period;
- new clients or content types appear that were not represented in training;
- analytics coverage changes materially, especially GA4 availability;
- the model has not been refreshed for a new reporting period;
- business rules or content-review priorities change.

The current model should therefore be treated as a versioned decision-support system rather than a permanent scoring rule.

In [15]:
# Current validation reference values

monitoring_reference = pd.DataFrame({
    "Metric": [
        "Validation accuracy",
        "Validation precision",
        "Validation recall",
        "Validation F1",
        "Review First share",
        "Monitor share",
        "Lower Priority share"
    ],
    "Reference Value": [
        0.7271,
        0.5798,
        0.1271,
        0.2085,
        (test_df["recommended_action"] == "Review First").mean(),
        (test_df["recommended_action"] == "Monitor").mean(),
        (test_df["recommended_action"] == "Lower Priority").mean()
    ]
})

monitoring_reference["Reference Value"] = (
    monitoring_reference["Reference Value"]
    .astype(float)
    .round(4)
)

display(monitoring_reference)


,Metric,Reference Value
0,Validation accuracy,0.7271
1,Validation precision,0.5798
2,Validation recall,0.1271
3,Validation F1,0.2085
4,Review First share,0.2434
5,Monitor share,0.3473
6,Lower Priority share,0.4093


In [16]:
# Example retrain / review triggers

retrain_triggers = pd.DataFrame({
    "Trigger": [
        "Precision drops below 0.45",
        "Recall drops below 0.08",
        "Action-category share shifts by > 15 percentage points",
        "Major feature distribution shift",
        "New client/content population",
        "Large change in analytics coverage",
        "New reporting period available"
    ],
    "Response": [
        "Review threshold and retrain",
        "Audit missed decline cases and retrain",
        "Check score calibration and data drift",
        "Compare current vs training distributions",
        "Revalidate on the new population",
        "Reassess engagement features",
        "Retrain using a newer time window"
    ]
})

display(retrain_triggers)

,Trigger,Response
0,Precision drops below 0.45,Review threshold and retrain
1,Recall drops below 0.08,Audit missed decline cases and retrain
2,Action-category share shifts by > 15 percentag...,Check score calibration and data drift
3,Major feature distribution shift,Compare current vs training distributions
4,New client/content population,Revalidate on the new population
5,Large change in analytics coverage,Reassess engagement features
6,New reporting period available,Retrain using a newer time window


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exported Artifacts

The ranked recommendation queue is exported so it can be reused in the capstone paper and reviewed outside the notebook.

The exported file contains:

- priority rank;
- anonymous client and content identifiers;
- decline-risk score;
- recommended action;
- human-readable reason codes;
- historical search and engagement signals used to support interpretation.

The export contains no client names, domains, URLs, private queries, or credentials.

In [17]:
from pathlib import Path

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

queue_export = ranked_queue.reset_index()

queue_path = output_dir / "w07_ranked_action_queue.csv"

queue_export.to_csv(
    queue_path,
    index=False
)

print("Saved:", queue_path)
print("Rows exported:", len(queue_export))
print("Columns exported:", len(queue_export.columns))

display(queue_export.head(10))

Saved: ../outputs/w07_ranked_action_queue.csv
Rows exported: 31824
Columns exported: 10


,priority_rank,client_hash_id,content_hash_id,decline_risk_score,recommended_action,reason_codes,past_daily_impressions,past_ctr,past_avg_position,past_engagement_rate
0,1,client_3f0ce4d44fe94f3d,content_e186467e031ce88d,0.78674,Review First,LOW_CLICK_CAPTURE | WEAK_SEARCH_POSITION | LOW...,0.052632,0.0,29.0,0.0
1,2,client_3f0ce4d44fe94f3d,content_d16ccdd45b7c48f7,0.78674,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,10.0,0.0
2,3,client_3f0ce4d44fe94f3d,content_ec8fd442962b9bce,0.78674,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,10.0,0.0
3,4,client_3f0ce4d44fe94f3d,content_25d7a0a4aaca8746,0.78674,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,11.0,0.0
4,5,client_3f0ce4d44fe94f3d,content_8c087c4643248bc2,0.78674,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,9.0,0.0
5,6,client_3f0ce4d44fe94f3d,content_cf88a82c77fd99ad,0.78674,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,9.0,0.0
6,7,client_3f0ce4d44fe94f3d,content_39cac2a87e36bac3,0.78674,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,11.0,0.0
7,8,client_3f0ce4d44fe94f3d,content_a905934677d7d92e,0.78674,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,11.0,0.0
8,9,client_3f0ce4d44fe94f3d,content_546d4dbcb6ac7121,0.78674,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,9.0,0.0
9,10,client_3f0ce4d44fe94f3d,content_c45fe9b1aaa86b88,0.78674,Review First,LOW_CLICK_CAPTURE | LOW_ENGAGEMENT | DECLINE_R...,0.052632,0.0,9.0,0.0


In [18]:
# Export a compact summary for the paper

action_summary = (
    test_df["recommended_action"]
    .value_counts()
    .rename_axis("recommended_action")
    .reset_index(name="pages")
)

action_summary["share_pct"] = (
    action_summary["pages"] /
    action_summary["pages"].sum() * 100
).round(2)

summary_path = output_dir / "w07_action_summary.csv"

action_summary.to_csv(
    summary_path,
    index=False
)

print("Saved:", summary_path)

display(action_summary)

Saved: ../outputs/w07_action_summary.csv


,recommended_action,pages,share_pct
0,Lower Priority,13025,40.93
1,Monitor,11054,34.73
2,Review First,7745,24.34


In [19]:
# Verify exported files exist and are non-empty

export_check = pd.DataFrame({
    "File": [
        str(queue_path),
        str(summary_path)
    ],
    "Exists": [
        queue_path.exists(),
        summary_path.exists()
    ],
    "Size bytes": [
        queue_path.stat().st_size if queue_path.exists() else 0,
        summary_path.stat().st_size if summary_path.exists() else 0
    ]
})

display(export_check)

,File,Exists,Size bytes
0,../outputs/w07_ranked_action_queue.csv,True,5754270
1,../outputs/w07_action_summary.csv,True,106


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.